# Schönfinkel 1924：去变量化的现代教学复现

这不是原论文附带的程序，也不是对其全部逻辑系统的机械化证明。它用纯 Python 标准库演示五个组合子的约化、现代括号抽象算法、有限测试和量词的有限域语义。

原论文记号是 I、C、T、Z、S。本项目以原记号为主：原 C = 现代 K；原 T 常对应现代 C；原 Z 常对应现代 B。Python 中 K 只是 C 的别名，因此打印输出使用 C。

执行前：将工作目录设为本 notebook 所在文件夹。交付版输出由 execute_notebook.py 按顺序执行普通 Python 单元格产生；没有使用 Jupyter 内核，也不假装拥有内核执行记录。


In [1]:
from combinators import (Var, App, I, C, K, T, Z, S, app, step, normalize,
                         abstract, substitute, free_variables, finite_u,
                         StepLimitExceeded)
f, g, x, y = map(Var, ('f', 'g', 'x', 'y'))
print('现代 K 与原 C 是同一个常量：', K is C)
print('应用左结合：', app(f, x, y))


现代 K 与原 C 是同一个常量： True
应用左结合： ((f x) y)


## 1. 五条约化规则

I x → x；C x y → x；T f x y → f y x；Z f g x → f (g x)；S f g x → f x (g x)。

AST 是不可变的 Var / Comb / App。一步约化按最左最外顺序选择 redex；参数不足不触发规则。完全正规化也会进入应用的参数，而非只求弱头正规形。


In [2]:
examples = [app(I,x), app(C,x,y), app(T,f,x,y), app(Z,f,g,x), app(S,f,g,x)]
for term in examples:
    print(str(term), '→', str(step(term)))
assert step(app(C,x)) is None
assert step(app(C,x,y,g)) == app(x,g)
print('参数不足与额外参数：通过')


(I x) → x
((C x) y) → x
(((T f) x) y) → ((f y) x)
(((Z f) g) x) → (f (g x))
(((S f) g) x) → ((f x) (g x))
参数不足与额外参数：通过


## 2. 现代括号抽象：把变量变成参数

定义 [x]x = I；若 x 不自由出现于 E，则 [x]E = C E；其余应用情形为 [x](P Q) = S ([x]P) ([x]Q)。这里 C 就是现代 K。

目标性质是 ([x]E) N 与 E[x := N] 可约化到相同结果。一般结论需要结构归纳论证；后面的有限穷举只能发现这个实现中的部分错误。本 AST 没有 λ 绑定器，替换不会捕获变量。


In [3]:
expression = app(f,x,app(g,x))
compiled = abstract('x',expression)
left, count = normalize(app(compiled,y))
right, _ = normalize(substitute(expression,'x',y))
print('E =', expression)
print('[x]E =', compiled)
print('应用到 y 后 =', left, '；步数 =', count)
assert left == right == app(f,y,app(g,y))
assert 'x' not in free_variables(compiled)
print('自由变量：', ', '.join(sorted(free_variables(compiled))))


E = ((f x) (g x))
[x]E = ((S ((S (C f)) I)) ((S (C g)) I))
应用到 y 后 = ((f y) (g y)) ；步数 = 7
自由变量： f, g


## 3. 原论文的消去恒等式

验证 I = S C C，Z = S (C S) C，T = S (Z Z S) (C C)。右边并不是与左边相同的 AST；我们给两边补足新鲜自由变量参数，然后比较正规形。这是这些等式的符号演示，不是对所有 Python 函数或任意无穷约化的外延等价判定程序。

不把原文涉及 φ 的更一般写法误说成无限制的全函数定理。


In [4]:
identities = [('I',I,app(S,C,C),(x,)),
              ('Z',Z,app(S,app(C,S),C),(f,g,x)),
              ('T',T,app(S,app(Z,Z,S),app(C,C)),(f,x,y))]
for name, primitive, derived, args in identities:
    direct, _ = normalize(app(primitive,*args))
    expanded, count = normalize(app(derived,*args))
    assert direct == expanded
    print(name, '=', derived, '；补足参数后 →', expanded, '；展开侧步数', count)


I = ((S C) C) ；补足参数后 → x ；展开侧步数 2
Z = ((S (C S)) C) ；补足参数后 → (f (g x)) ；展开侧步数 4
T = ((S ((Z Z) S)) (C C)) ；补足参数后 → ((f y) x) ；展开侧步数 6


## 4. 发散与步数预算

令 W = S I I，Ω = W W。W 把参数用于自身，因此 Ω 存在重复展开的非终止行为。程序只报告预算耗尽，不从一个超时一般性地断言发散。

C x Ω 可在一步丢弃 Ω；这体现最左最外顺序的作用。预算是约化步数上限，不是内存、递归深度或墙钟时间的保护。


In [5]:
w = app(S,I,I)
omega = app(w,w)
try:
    normalize(omega,max_steps=25)
except StepLimitExceeded as error:
    print(str(error))
else:
    raise AssertionError('Expected the reduction budget to be exhausted')
value, count = normalize(app(C,x,omega),max_steps=1)
assert value == x and count == 1
print('C x Ω →', value, '；步数 =', count)


Reduction budget exhausted after 25 steps; this alone does not prove divergence.
C x Ω → x ；步数 = 1


## 5. U 的有限域语义演示

将 U(f)(g) 解释为 ∀x ¬(f(x) ∧ g(x))：两个谓词没有共同成立的对象。这里用非空有限域 {0,1,2} 上的纯、全定义布尔谓词计算；U 不属于上面五个组合子的语法约化规则。

它不是逐点 NAND：外层有对整个域的全称量化。本教学 API 特意拒绝空域；这不是说现代逻辑中的空域全称命题不能讨论。也没有声称有限枚举实现了原论文中任意对象域上的量化。


In [6]:
domain = (0,1,2)
u = finite_u(domain)
even = lambda n: n % 2 == 0
odd = lambda n: n % 2 == 1
is_two = lambda n: n == 2
print('偶数与奇数没有共同元素：', u(even)(odd))
print('偶数与等于 2 没有共同元素：', u(even)(is_two))
assert u(even)(odd) is True
assert u(even)(is_two) is False


偶数与奇数没有共同元素： True
偶数与等于 2 没有共同元素： False


## 6. 让 U 的量化域变为「谓词集合」

设对象域 D={0,1,2}，P 为 D 上选定的布尔谓词集合。分别构造 U_D 与 U_P；这两个闭包的量化域不同。

定义 A(f)=U_P(U_D(f))(U_D(f))。因为 U_D(f) 是关于谓词 g 的谓词，所以 A(f)=¬∃g∈P U_D(f)(g)。进一步 U_P(A)(A)=∀f∈P ∃g∈P U_D(f)(g)。

这是一种有层次、显式指定域的有限语义演示，**不是把同一个 U_D 闭包直接应用于自身**。若 P 包含全部八个谓词，可选恒假 g，结果为真；若 P 只包含恒真谓词，结果为假。这也说明量词域的选择会改变命题的真假。


In [7]:
all_predicates = tuple(lambda x,mask=mask: bool(mask & (1 << x)) for mask in range(8))
restricted_predicates = (lambda x: True,)
for label, predicates, expected in [('全部八个谓词',all_predicates,True),
                                     ('仅恒真谓词',restricted_predicates,False)]:
    u_objects = finite_u(domain)
    u_predicates = finite_u(predicates)
    def a(predicate):
        incompatibility = u_objects(predicate)
        return u_predicates(incompatibility)(incompatibility)
    for predicate in predicates:
        assert a(predicate) == (not any(u_objects(predicate)(other) for other in predicates))
    encoded = u_predicates(a)(a)
    direct = all(any(u_objects(predicate)(other) for other in predicates) for predicate in predicates)
    assert encoded == direct == expected
    print(label, '：U 编码 =', encoded, '；直接 ∀f∃g 枚举 =', direct)


全部八个谓词 ：U 编码 = True ；直接 ∀f∃g 枚举 = True
仅恒真谓词 ：U 编码 = False ；直接 ∀f∃g 枚举 = False


## 7. 实际运行测试

替换性质穷举的精确边界：叶子来自 {x,y,I,C,S}；AST 总节点数为 1、3 或 5 的全部 280 个表达式；替换项来自 {y,I,C,I y,C y}，共 1,400 对。它并不穷举所有项或所有替换。

量词部分枚举三元素域上的全部 8 个布尔谓词，测试全部 64 对。其余测试覆盖不可变性、规则、部分应用、额外参数、约化顺序、无捕获和预算边界。


In [8]:
import unittest
import test_combinators
suite = unittest.defaultTestLoader.loadTestsFromModule(test_combinators)
result = unittest.TextTestRunner(verbosity=1).run(suite)
assert result.wasSuccessful()
print('测试方法数：', result.testsRun)


Exhaustive finite substitution checks: 280 expressions × 5 replacements = 1400
Finite U checks: all 8 × 8 Boolean-predicate pairs on domain {0,1,2} = 64
Higher-order U: predicate domain all eight; encoded=True; direct=True
Higher-order U: predicate domain only constant True; encoded=False; direct=False
Original identities verified after fresh-variable arguments: I=SCC; Z=S(CS)C; T=S(ZZS)(CC)
测试方法数： 11


...........
----------------------------------------------------------------------
Ran 11 tests in 0.038s

OK


## 限制与阅读定位

- 这是离散语法演算的教学模型，不是原论文的数字化全文、实验代码或完整定理证明。
- 本版本不实现解析器、类型系统、λ AST、α 改名、无界量化、完全性/一致性证明或通用等价判定。
- 递归 AST 操作可能达到 Python 递归深度限制；S 会复制项并产生很大树。不要将步数预算误认为资源隔离。
- 所有例子都没有网络请求、随机性、凭证或第三方依赖。具体命令、环境与实际日志见 README.md 和 logs/。
